# AI-RAG · Week 11 — Notebook 2
## Eval as Code & the 8-KPI Dashboard

This notebook assembles the **8-KPI dashboard** from scored rows, persists per-query cost/latency to `results.db` tagged by `kb_version`, renders the baseline report, and sets **thresholds from a baseline**. Runs **offline** end-to-end. It reuses the metric machinery from Notebook 1 — included below under *Prerequisites* so this notebook runs on its own — then spends its narration on what's new in Day 2.

> Code is verbatim from `src/eval/*.py` and `scripts/*.py`; only notebook glue (paths, inlined imports, and replacing the `argparse` CLI wrapper with direct cells) differs.

### Setup

In [1]:
# --- notebook setup (delivery glue only) -------------------------------------
# 1) ensure pydantic is available   2) write the golden set locally
# 3) force the OFFLINE proxy path so this runs top-to-bottom with no API key/cost.
try:
    import pydantic  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pydantic"])

import os
from pathlib import Path
os.environ["EVAL_OFFLINE"] = "1"   # <- set an OPENAI_API_KEY and remove this line to run LIVE RAGAS

GOLDEN_JSONL = r"""{"id": "g001", "question": "What is the company's leave policy?", "ideal_answer": "Employees get 20 days of paid leave per year, accrued monthly at 1.67 days per month. Unused leave caps at 30 days; anything above that lapses on 31 December.", "notes": "happy-path. Source: HR-policy.pdf §3.1."}
{"id": "g002", "question": "How do I request a reimbursement for a work-related expense under $100?", "ideal_answer": "Submit the receipt in the expense portal within 30 days of the purchase. Items under $100 are auto-approved if they fit a standard category (travel, meals, supplies). Submission outside 30 days requires manager approval.", "notes": "happy-path. Source: Finance-policy.pdf §5.4."}
{"id": "g003", "question": "What is the standard working week?", "ideal_answer": "The standard working week is 40 hours, Monday to Friday, 09:00 to 17:30 with a one-hour lunch. Flexible start times between 08:00 and 10:00 are permitted with manager approval.", "notes": "happy-path. Source: HR-policy.pdf §1.2."}
{"id": "g004", "question": "What's the policy on bringing personal devices to work (BYOD)?", "ideal_answer": "Personal devices may be used to access company email and approved SaaS apps via the MDM enrollment. Devices must be password-protected, encrypted, and patched within 30 days of OS updates. Local storage of company documents is prohibited.", "notes": "happy-path. Source: IT-security.pdf §2.3."}
{"id": "g005", "question": "How do I request parental leave?", "ideal_answer": "Submit a parental leave request via the HR portal at least 8 weeks before the expected leave start date. Primary caregivers are entitled to 16 weeks paid; secondary caregivers to 4 weeks paid. Additional unpaid leave up to 12 months may be granted with manager approval.", "notes": "happy-path. Source: HR-policy.pdf §3.4."}
{"id": "g006", "question": "What's the dress code for client meetings?", "ideal_answer": "Business smart for in-person client meetings: collared shirt or blouse, smart trousers or skirt, closed-toe shoes. Jeans and sneakers are acceptable for internal meetings but not for client-facing ones.", "notes": "happy-path. Source: HR-policy.pdf §2.1."}
{"id": "g007", "question": "How do I report a security incident?", "ideal_answer": "Email security@company.com immediately and create a ticket in the IT portal flagged as 'Security Incident'. For incidents involving customer data, also notify the Data Protection Officer within 24 hours. Do not investigate the incident yourself — preserve evidence.", "notes": "happy-path. Source: IT-security.pdf §4.1."}
{"id": "g008", "question": "What's the process for requesting new software?", "ideal_answer": "Software requests go through the IT portal. Pre-approved software (listed in the IT catalogue) is auto-installed within 1 business day. New requests require justification, a security review (2-5 business days), and budget approval if the cost exceeds $500/year.", "notes": "happy-path. Source: IT-policy.pdf §3.2."}
{"id": "g009", "question": "Who do I contact about my pay or benefits?", "ideal_answer": "For pay queries, contact Payroll at payroll@company.com or via the HR portal under 'Pay & Benefits'. For benefits (health insurance, retirement, etc.), contact HR Benefits at benefits@company.com.", "notes": "happy-path. Source: HR-policy.pdf §6."}
{"id": "g010", "question": "What's the policy on working from home?", "ideal_answer": "Employees may work remotely up to 3 days per week with manager approval. Remote work requires a stable internet connection, a quiet workspace, and availability during core hours (10:00-15:00 local time). Full-time remote arrangements require VP-level approval.", "notes": "happy-path. Source: HR-policy.pdf §1.5."}
{"id": "g011", "question": "How do I book a meeting room?", "ideal_answer": "Meeting rooms are booked via the Outlook calendar. Rooms over 12 people require booking at least 24 hours in advance. Recurring meetings should not block premium rooms (those with video conferencing) for more than 2 hours per week.", "notes": "happy-path. Source: Facilities-policy.pdf §1.1."}
{"id": "g012", "question": "What's the company's policy on accepting gifts from clients?", "ideal_answer": "Gifts under $50 in value may be accepted with line-manager notification. Gifts above $50 must be declined or, if culturally inappropriate to decline, donated to charity with HR notified. Cash, gift cards, and personal services may never be accepted.", "notes": "happy-path. Source: Compliance-policy.pdf §2.1."}
{"id": "g013", "question": "How do I onboard a new contractor?", "ideal_answer": "Submit a contractor onboarding request via the HR portal at least 5 business days before the start date. Include scope of work, contract duration, budget code, and the contractor's NDA status. IT provisioning takes 2 business days from approval.", "notes": "happy-path. Source: HR-policy.pdf §5.1."}
{"id": "g014", "question": "What training is required for new employees?", "ideal_answer": "All new employees complete three mandatory trainings within 30 days of joining: (1) Information Security & Data Protection, (2) Code of Conduct & Anti-Harassment, (3) Health & Safety. Tracked via the L&D portal; manager is notified if not completed by day 30.", "notes": "happy-path. Source: HR-policy.pdf §4.2."}
{"id": "g015", "question": "How does the remote work policy interact with the requirement to attend in-person team meetings?", "ideal_answer": "Remote work allowance (up to 3 days per week) is subject to attending mandatory in-person team meetings as scheduled by the manager. Teams typically schedule one anchor in-person day per week. Recurring conflicts should be resolved by the team's manager, who may approve exceptions on a case-by-case basis.", "notes": "harder · multi-hop. Cross-references HR-policy §1.5 (remote) AND §2.2 (team meetings). Tests integration across two source sections."}
{"id": "g016", "question": "If I'm on parental leave and a serious security incident happens in my product area, am I expected to respond?", "ideal_answer": "No. Employees on parental leave are not expected to respond to work matters, including security incidents. The on-call rotation and the team's secondary contact (named in each team's runbook) cover incident response during any team member's absence. If the matter is critical and only the absent employee has the knowledge, the line manager escalates to the VP for coverage.", "notes": "harder · multi-hop. Cross-references HR-policy §3.4 (parental leave) AND IT-security §4.1 (incident response). Tests respect for leave + named escalation path."}
{"id": "g017", "question": "What's the right channel to report a potential conflict of interest involving a vendor my partner works for?", "ideal_answer": "Disclose the potential conflict via the Compliance portal under 'Conflict of Interest Disclosure'. Include the vendor name, your partner's role, and any business relationship you have with the vendor. The Compliance team reviews disclosures within 5 business days and may require recusal from procurement or vendor-management decisions involving that vendor.", "notes": "harder. Combines Compliance-policy §3 (disclosures) and process detail. Tests nuanced conflict-of-interest reporting."}
{"id": "g018", "question": "Can I expense a Coursera certification that's not in the L&D catalogue if it directly supports my project?", "ideal_answer": "Yes, with manager pre-approval. Submit a request via the L&D portal under 'Out-of-catalogue learning' with a one-paragraph business justification. Certifications up to $1,000 per year per employee are typically approved for project-relevant skills. Reimbursement requires successful completion (certificate issued) within 6 months.", "notes": "harder. Cross-references L&D-policy §2.4 and Finance §5. Tests handling of off-catalogue requests with conditions."}
{"id": "g019", "question": "What is the company's stance on cryptocurrency investing as part of the benefits package?", "ideal_answer": "Cryptocurrency investing is not part of the benefits package. The answer is not in the company's policy documents. The assistant should say so clearly and suggest the employee contact HR for clarification.", "notes": "edge. Tests graceful refusal — answer NOT in docs. Should not hallucinate."}
{"id": "g020", "question": "Will the company sponsor my visa for a personal trip to a country requiring a business invitation letter?", "ideal_answer": "Visa support is only provided for company-sanctioned business travel — see Travel-policy §3.2. Personal travel visas are the employee's responsibility. The company does not issue invitation letters for personal trips.", "notes": "edge. Tests boundary-setting — adjacent to a real policy but clearly outside its scope. Should redirect, not invent."}"""
Path("golden_set").mkdir(exist_ok=True)
Path("golden_set/capstone_v1.jsonl").write_text(GOLDEN_JSONL + "\n")
print("golden set written:", len(GOLDEN_JSONL.splitlines()), "entries · EVAL_OFFLINE =", os.environ["EVAL_OFFLINE"])


golden set written: 20 entries · EVAL_OFFLINE = 1


### Prerequisites from Notebook 1

You met these in Notebook 1 — the golden loader, the four metric functions, and the row-builder. They're re-established here so this notebook is self-contained; the new material starts at section 1.

In [2]:
"""src/eval/golden.py — loader and validator for the capstone golden set.

Lineage: carried forward unchanged from W5. The golden set is the ground truth
that every later week's KPI is computed against — treat it as infrastructure,
not a one-off fixture. As of W10 it lives at `golden_set/capstone_v1.jsonl`
(the W5 set plus W7/W9 additions), 20 entries, ids g001..g020.

A golden entry has four fields:
  • id            — stable identifier (g001..g020). Referenced in eval runs
                    and across docs.
  • question      — exactly as a real user would phrase it.
  • ideal_answer  — full reference text. RAGAS uses this as `ground_truth`;
                    the judge uses it as the ideal answer. For open-ended
                    questions this may be a properties dict.
  • notes         — category (happy/harder/edge), source doc, why it matters.
"""
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any

from pydantic import BaseModel, Field, field_validator

_ID_PATTERN = re.compile(r"^g\d{3}$")


class GoldenEntry(BaseModel):
    """One row of the golden set."""

    id: str = Field(..., description="Stable identifier g001..g999")
    question: str = Field(..., min_length=1)
    ideal_answer: str | dict[str, Any] = Field(...)
    notes: str = Field(default="", description="Category, source, why this matters")

    @field_validator("id")
    @classmethod
    def _id_shape(cls, v: str) -> str:
        if not _ID_PATTERN.match(v):
            raise ValueError(f"id must match g001..g999; got {v!r}")
        return v

    @property
    def ground_truth(self) -> str:
        """RAGAS expects a string ground_truth. Flatten dict ideals to text."""
        if isinstance(self.ideal_answer, dict):
            return json.dumps(self.ideal_answer)
        return self.ideal_answer


def load_golden_set(path: str | Path) -> list[GoldenEntry]:
    """Read a .jsonl file. Each non-blank, non-comment line is one entry."""
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Golden set not found at {p}")
    entries: list[GoldenEntry] = []
    with p.open() as f:
        for lineno, raw in enumerate(f, start=1):
            line = raw.strip()
            if not line or line.startswith("//"):
                continue
            try:
                data = json.loads(line)
            except json.JSONDecodeError as e:
                raise ValueError(f"{p}:{lineno} — malformed JSON: {e.msg}") from e
            entries.append(GoldenEntry.model_validate(data))
    return entries


def validate_golden_set(path: str | Path) -> dict[str, Any]:
    """Validate the file. Returns a summary dict; raises on errors."""
    entries = load_golden_set(path)

    ids = [e.id for e in entries]
    if len(ids) != len(set(ids)):
        seen: set[str] = set()
        dupes = [i for i in ids if i in seen or seen.add(i)]  # noqa: B015
        raise ValueError(f"Duplicate ids in {path}: {sorted(set(dupes))}")

    def _bucket(note: str) -> str:
        low = note.lower()
        if low.startswith("happy"):
            return "happy"
        if low.startswith("harder") or low.startswith("multi"):
            return "harder"
        if low.startswith("edge"):
            return "edge"
        return "uncategorised"

    counts: dict[str, int] = {}
    for e in entries:
        counts[_bucket(e.notes)] = counts.get(_bucket(e.notes), 0) + 1

    return {"path": str(path), "n": len(entries), "ids": ids, "coverage": counts}


In [3]:
"""src/eval/ragas_metrics.py — the four RAG-specific metrics, two ways to run.

W11 standardises on RAGAS for the four RAG metrics:
  • faithfulness        — is the answer supported by the retrieved context?
  • answer_relevance    — does the answer address the question?
  • context_precision   — are the top-K retrieved chunks the right ones?
  • context_recall      — did we retrieve everything we needed?

Each public function takes a list of `rows` and returns a `MetricResult`
(per-row scores + an aggregate). A row is a dict with:
    {"id", "question", "answer", "contexts": [str, ...], "ground_truth"}

TWO MODES
---------
LIVE  (default when an API key is set and `ragas` is importable):
      real RAGAS metrics, each backed by an LLM judge call per row. This is
      what produces the numbers you put in the snapshot and defend at M2.

OFFLINE (set EVAL_OFFLINE=1, or when `ragas` isn't installed):
      deterministic token-overlap *proxies*. These are NOT RAGAS numbers —
      they exist so the eval suite runs in CI / on a laptop with no API key,
      proving the wiring is sound. Every offline MetricResult is labelled
      `mode="offline-proxy"` so a proxy number is never mistaken for a real
      RAGAS score. The smoke test runs this path.

This split is the whole point of "eval as code": the suite is runnable and
assertable on every commit (offline), and produces real numbers on demand
(live). See docs/eval/README.md.
"""
from __future__ import annotations

import os
import re
from dataclasses import dataclass, field


# ─── Result container ────────────────────────────────────────────────────────
@dataclass
class MetricResult:
    """One metric's outcome across all rows."""

    name: str
    mode: str                       # "live" or "offline-proxy"
    aggregate: float                # mean across rows, 0..1
    per_row: dict[str, float] = field(default_factory=dict)  # id -> score

    def __str__(self) -> str:
        return f"{self.name}={self.aggregate:.3f} ({self.mode}, n={len(self.per_row)})"


# ─── Mode detection ──────────────────────────────────────────────────────────
def offline_mode() -> bool:
    """Offline if explicitly forced, or if ragas / an API key is unavailable."""
    if os.getenv("EVAL_OFFLINE", "").strip() in {"1", "true", "TRUE", "yes"}:
        return True
    if not os.getenv("OPENAI_API_KEY"):
        return True
    try:
        import ragas  # noqa: F401
    except Exception:
        return True
    return False


# ─── Token helpers (offline proxy) ───────────────────────────────────────────
_STOP = {
    "the", "a", "an", "and", "or", "but", "if", "of", "to", "in", "on", "for",
    "with", "is", "are", "was", "were", "be", "been", "as", "at", "by", "it",
    "this", "that", "these", "those", "from", "your", "you", "may", "must",
    "can", "do", "does", "how", "what", "who", "i", "my", "me", "we", "our",
    "per", "up", "any", "all", "via", "not", "no", "they", "their",
}


def _tokens(text: str) -> set[str]:
    if not text:
        return set()
    raw = re.findall(r"[a-z0-9]+", text.lower())
    return {t for t in raw if len(t) > 2 and t not in _STOP}


def _overlap_fraction(a: set[str], b: set[str]) -> float:
    """Fraction of `a`'s tokens that appear in `b`. 0..1. Empty `a` -> 0.0."""
    if not a:
        return 0.0
    return len(a & b) / len(a)


# ─── Offline proxies (deterministic, dependency-free) ────────────────────────
def _proxy(name: str, rows: list[dict], fn) -> MetricResult:
    per_row = {r["id"]: round(fn(r), 4) for r in rows}
    agg = round(sum(per_row.values()) / len(per_row), 4) if per_row else 0.0
    return MetricResult(name=name, mode="offline-proxy", aggregate=agg, per_row=per_row)


def _faithfulness_proxy(r: dict) -> float:
    # How much of the answer is covered by the retrieved context?
    ans = _tokens(r.get("answer", ""))
    ctx = _tokens(" ".join(r.get("contexts", [])))
    return _overlap_fraction(ans, ctx)


def _answer_relevance_proxy(r: dict) -> float:
    # Does the answer address the question? Blend (a) how many of the question's
    # key terms the answer covers with (b) whether the answer engages the topic
    # at all. Short questions make pure token-overlap jumpy, so the topical-touch
    # term stabilises a correct answer near ~0.7 while a truly off-topic answer
    # (no shared terms) still collapses to 0.0.
    q = _tokens(r.get("question", ""))
    ans = _tokens(r.get("answer", ""))
    if not q:
        return 0.0
    coverage = _overlap_fraction(q, ans)            # 0..1
    touches = 1.0 if (q & ans) else 0.0             # any shared key term?
    return round(0.5 * coverage + 0.5 * touches, 4)


def _context_precision_proxy(r: dict) -> float:
    # Of the retrieved chunks, what fraction are relevant to the ground truth?
    gt = _tokens(r.get("ground_truth", ""))
    contexts = r.get("contexts", []) or []
    if not contexts:
        return 0.0
    relevant = sum(1 for c in contexts if _tokens(c) & gt)
    return relevant / len(contexts)


def _context_recall_proxy(r: dict) -> float:
    # How much of the ground truth is present across the retrieved chunks?
    gt = _tokens(r.get("ground_truth", ""))
    ctx = _tokens(" ".join(r.get("contexts", [])))
    return _overlap_fraction(gt, ctx)


# ─── Live RAGAS ──────────────────────────────────────────────────────────────
def _ragas_score(name: str, rows: list[dict], metric, column: str) -> MetricResult:
    """Run one RAGAS metric over the rows. Imported lazily (heavy deps).

    Targets ragas ~0.1.x (see requirements.txt for the pin). RAGAS has changed
    column names and metric imports across versions — if this raises, check the
    pinned version and the Lab Guide's 'RAGAS version pinning' note.
    """
    from datasets import Dataset
    from ragas import evaluate

    ds = Dataset.from_dict({
        "question":     [r["question"] for r in rows],
        "answer":       [r["answer"] for r in rows],
        "contexts":     [list(r.get("contexts", [])) for r in rows],
        "ground_truth": [r.get("ground_truth", "") for r in rows],
    })
    result = evaluate(ds, metrics=[metric])
    df = result.to_pandas()

    per_row = {
        rows[i]["id"]: round(float(df[column].iloc[i]), 4)
        for i in range(len(rows))
    }
    agg = round(float(df[column].mean()), 4)
    return MetricResult(name=name, mode="live", aggregate=agg, per_row=per_row)


# ─── Public API — one function per metric ────────────────────────────────────
def score_faithfulness(rows: list[dict]) -> MetricResult:
    if offline_mode():
        return _proxy("faithfulness", rows, _faithfulness_proxy)
    from ragas.metrics import faithfulness
    return _ragas_score("faithfulness", rows, faithfulness, "faithfulness")


def score_answer_relevance(rows: list[dict]) -> MetricResult:
    if offline_mode():
        return _proxy("answer_relevance", rows, _answer_relevance_proxy)
    from ragas.metrics import answer_relevancy
    return _ragas_score("answer_relevance", rows, answer_relevancy, "answer_relevancy")


def score_context_precision(rows: list[dict]) -> MetricResult:
    if offline_mode():
        return _proxy("context_precision", rows, _context_precision_proxy)
    from ragas.metrics import context_precision
    return _ragas_score("context_precision", rows, context_precision, "context_precision")


def score_context_recall(rows: list[dict]) -> MetricResult:
    if offline_mode():
        return _proxy("context_recall", rows, _context_recall_proxy)
    from ragas.metrics import context_recall
    return _ragas_score("context_recall", rows, context_recall, "context_recall")


# Convenience: all four at once (used by run_eval.py and the baseline doc).
ALL_METRICS = (
    score_faithfulness,
    score_answer_relevance,
    score_context_precision,
    score_context_recall,
)


def score_all(rows: list[dict]) -> dict[str, MetricResult]:
    results = [fn(rows) for fn in ALL_METRICS]   # one call each (matters live!)
    return {r.name: r for r in results}


In [4]:
"""src/eval/harness.py — build the eval rows once, two ways.

Shared by eval/conftest.py (the pytest suite) and scripts/run_eval.py (the CLI),
so the offline synthetic pipeline and the live capstone pipeline are defined in
exactly one place.

Row shape (identical offline and live):
    {"id", "question", "answer", "contexts": [str,...], "ground_truth",
     "cache_hit", "latency_ms", "cost_usd", "notes"}
"""
from __future__ import annotations

from pathlib import Path

# GoldenEntry, load_golden_set, offline_mode are defined in the cells above

REPO_ROOT = Path.cwd()   # notebook: resolve paths from the working directory
GOLDEN_PATH = REPO_ROOT / "golden_set" / "capstone_v1.jsonl"

# KB version active at eval time. Live mode reads it from the pipeline; offline
# pins the W10 value so every eval run and snapshot is tagged by KB state (D4).
OFFLINE_KB_VERSION = "v2.w10"

# A clean distractor sharing almost no content tokens with policy answers, so
# context precision reflects relevant-vs-retrieved honestly.
_DISTRACTOR = (
    "The office kitchen is restocked every Monday morning and the building "
    "reception desk is open from eight until six."
)


def _split_two(text: str) -> tuple[str, str]:
    words = text.split()
    mid = max(1, len(words) // 2)
    return " ".join(words[:mid]), " ".join(words[mid:])


def _synth_row(entry: GoldenEntry, index: int) -> dict:
    """Deterministically build one (answer, contexts) pair for a golden entry.

    Most rows are faithful and complete. A few are deliberately degraded so the
    report has real signal: g019 hallucinates, g015/g005 suffer retrieval misses.
    """
    gt = entry.ground_truth
    c1, c2 = _split_two(gt)
    contexts = [c1, c2, _DISTRACTOR]   # 2 relevant + 1 distractor
    answer = gt

    if entry.id == "g019":
        answer = gt + (" The company also matches up to five percent of salary "
                       "into a managed crypto retirement fund.")   # hallucination
    elif entry.id == "g015":
        contexts = [c1, _DISTRACTOR]                                # multi-hop miss
    elif entry.id == "g005":
        contexts = [c1[: max(1, len(c1) // 2)], _DISTRACTOR]        # weak retrieval

    cache_hit = index < 13          # ~65% warm hit rate, matching wk10
    return {
        "id":           entry.id,
        "question":     entry.question,
        "answer":       answer,
        "contexts":     contexts,
        "ground_truth": gt,
        "cache_hit":    cache_hit,
        "latency_ms":   184 if cache_hit else 785,
        "cost_usd":     0.000041 if cache_hit else 0.000412,
        "notes":        entry.notes,
    }


def _live_rows(golden: list[GoldenEntry]) -> list[dict]:  # pragma: no cover - live only
    """Run the real capstone over the golden set. Needs the learner's src/rag."""
    try:
        from src.rag.qdrant_rag import ask_rag
        from src.rag.qdrant_store import QdrantStore
        from src.rag.retrieval import (
            build_bm25_index,
            hybrid_retrieve,
            load_chunks_from_qdrant,
        )
    except Exception as exc:
        raise RuntimeError(
            "Live eval needs your capstone code (src/rag/*) on the path and "
            "OPENAI_API_KEY + QDRANT_URL set. Run offline with EVAL_OFFLINE=1. "
            f"Import error: {exc}"
        ) from exc

    store = QdrantStore()
    chunks = load_chunks_from_qdrant(store)          # tombstone-filtered (W10)
    bm25 = build_bm25_index(chunks)
    rows: list[dict] = []
    for e in golden:
        res = ask_rag(e.question, store, k=3)
        # Full chunk text for RAGAS contexts comes from retrieval, not from
        # ask_rag's stripped citations (D3).
        retrieved = hybrid_retrieve(store, bm25, chunks, e.question, top_k=3)
        rows.append({
            "id":           e.id,
            "question":     e.question,
            "answer":       res["answer"],
            "contexts":     [c.get("text", "") for c in retrieved],
            "ground_truth": e.ground_truth,
            "cache_hit":    res.get("cache_hit", False),
            "latency_ms":   res.get("latency_ms", 0),
            "cost_usd":     res.get("cost_usd", 0.0),
            "notes":        e.notes,
        })
    return rows


def active_kb_version() -> str:
    if offline_mode():
        return OFFLINE_KB_VERSION
    try:  # pragma: no cover - live only
        from src.rag.pipeline import KB_VERSION
        return KB_VERSION
    except Exception:  # pragma: no cover
        return "unknown"


def load_golden() -> list[GoldenEntry]:
    return load_golden_set(GOLDEN_PATH)


def build_rows(golden: list[GoldenEntry] | None = None) -> list[dict]:
    """The (question, answer, contexts, ground_truth, ...) rows under test."""
    golden = golden or load_golden()
    if offline_mode():
        return [_synth_row(e, i) for i, e in enumerate(golden)]
    return _live_rows(golden)


In [5]:
rows = build_rows()
kb_version = active_kb_version()
print('rows:', len(rows), '| kb_version:', kb_version)

rows: 20 | kb_version: v2.w10


## 1 · The 8-KPI dashboard — `src/eval/kpi.py`

The M2 deliverable's engine. It scores the rows once, grades task success (the judge live; grounded-AND-relevant offline), and returns the full 8-KPI dashboard plus the W11 extensions: the `kb_version` tag and the **cache-hit vs cache-miss accuracy split** that honours the W10 tombstone promise. Two KPIs are honestly **N/A** (tool success → W13, escalation → W17).

In [6]:
"""src/eval/kpi.py — assemble the 8-KPI dashboard from scored rows.

This is the M2 deliverable's engine. It takes the scored eval rows and the
active kb_version and returns the full 8-KPI dashboard, plus the W11 extension
dimensions (cache-hit-rate and the cache-hit vs cache-miss accuracy split that
honours the W10 tombstone promise — D4).

The 8 KPIs (curriculum 'KPIs' sheet) and their W11 sources:
  1 Task success rate   — judge (live) / derived from grounded+relevant (offline)
  2 Grounded resp. rate  — RAGAS faithfulness
  3 Hallucination rate   — 1 - faithfulness (rough inverse)
  4 Retrieval hit rate   — RAGAS context precision
  5 Tool success rate    — N/A until W13 (no tools yet)
  6 Cost per query       — mean cost_usd (from results.db when present)
  7 Latency p50 / p95    — percentiles of latency_ms
  8 Escalation rate      — N/A until W17 (no HITL yet)
"""
from __future__ import annotations

# score_all, offline_mode are defined in the cells above

TOOL_SUCCESS_NA = "N/A (W13 — no tools yet)"
ESCALATION_NA = "N/A (W17 — no HITL yet)"

# KPI #1 task-success bars when derived offline (no judge available).
# Grounded bar is set just above the hallucinated row's faithfulness so a
# hallucination counts as a task failure; the relevant bar catches off-topic.
_OFFLINE_GROUNDED_BAR = 0.65
_OFFLINE_RELEVANT_BAR = 0.40


def _percentile(values: list[float], p: float) -> float:
    """Nearest-rank percentile. p in 0..100."""
    if not values:
        return 0.0
    s = sorted(values)
    k = max(0, min(len(s) - 1, round((p / 100.0) * (len(s) - 1))))
    return float(s[k])


def _task_success_per_row(rows, metrics) -> dict[str, bool]:
    """Per-row task success. Live: the W5 judge. Offline: grounded AND relevant."""
    faith = metrics["faithfulness"].per_row
    rel = metrics["answer_relevance"].per_row

    if offline_mode():
        return {
            r["id"]: (faith.get(r["id"], 0) >= _OFFLINE_GROUNDED_BAR
                      and rel.get(r["id"], 0) >= _OFFLINE_RELEVANT_BAR)
            for r in rows
        }

    # Live: grade with the carried-forward W5 judge.
    from .judge import judge_one_sync  # pragma: no cover - live only
    out: dict[str, bool] = {}
    for r in rows:  # pragma: no cover - live only
        score = judge_one_sync(r["question"], r["answer"], r["ground_truth"])
        out[r["id"]] = score.task_success
    return out


def _rate(flags: dict[str, bool]) -> float:
    return round(sum(flags.values()) / len(flags), 4) if flags else 0.0


def compute_kpis(rows: list[dict], kb_version: str) -> dict:
    """Return {'kpis', 'extras', 'metrics'} for the snapshot and the KPI test."""
    metrics = score_all(rows)                       # 4 RAGAS metrics, one pass
    faith = metrics["faithfulness"]
    cprec = metrics["context_precision"]

    success = _task_success_per_row(rows, metrics)

    costs = [r["cost_usd"] for r in rows]
    lats = [float(r["latency_ms"]) for r in rows]
    hits = [r for r in rows if r.get("cache_hit")]
    misses = [r for r in rows if not r.get("cache_hit")]

    kpis = {
        "1_task_success_rate":     _rate(success),
        "2_grounded_response_rate": faith.aggregate,
        "3_hallucination_rate":    round(1.0 - faith.aggregate, 4),
        "4_retrieval_hit_rate":    cprec.aggregate,
        "5_tool_success_rate":     TOOL_SUCCESS_NA,
        "6_cost_per_query_usd":    round(sum(costs) / len(costs), 6) if costs else 0.0,
        "7_latency_p50_ms":        _percentile(lats, 50),
        "7_latency_p95_ms":        _percentile(lats, 95),
        "8_escalation_rate":       ESCALATION_NA,
    }

    # W11 extension dimensions — kb_version tag + cache split (D4).
    hit_success = {r["id"]: success[r["id"]] for r in hits}
    miss_success = {r["id"]: success[r["id"]] for r in misses}
    extras = {
        "kb_version":         kb_version,
        "eval_mode":          faith.mode,
        "cache_hit_rate":     round(len(hits) / len(rows), 4) if rows else 0.0,
        "cache_hit_accuracy": _rate(hit_success),
        "cache_miss_accuracy": _rate(miss_success),
        "n":                  len(rows),
    }
    return {"kpis": kpis, "extras": extras, "metrics": metrics}


def reportable_count(kpis: dict) -> tuple[int, int]:
    """(numeric KPIs, N/A KPIs). Distinct KPI rows 1-8 (latency counts once)."""
    na = sum(1 for k, v in kpis.items() if isinstance(v, str) and v.startswith("N/A"))
    # KPIs 1-8, but latency is split across two keys -> count distinct rows.
    distinct = {k.split("_", 1)[0] for k in kpis}
    return len(distinct) - na, na


**Compute the dashboard:**

In [7]:
out = compute_kpis(rows, kb_version)
kpis, extras = out['kpis'], out['extras']

print('=== 8-KPI dashboard ===')
for k, v in kpis.items():
    print(f'  {k:28s} {v}')

print('\n=== W11 extensions ===')
for k, v in extras.items():
    print(f'  {k:20s} {v}')

numeric, na = reportable_count(kpis)
print(f'\nreportable: {numeric} numeric + {na} N/A')

=== 8-KPI dashboard ===
  1_task_success_rate          0.8
  2_grounded_response_rate     0.9252
  3_hallucination_rate         0.0748
  4_retrieval_hit_rate         0.6667
  5_tool_success_rate          N/A (W13 — no tools yet)
  6_cost_per_query_usd         0.000171
  7_latency_p50_ms             184.0
  7_latency_p95_ms             785.0
  8_escalation_rate            N/A (W17 — no HITL yet)

=== W11 extensions ===
  kb_version           v2.w10
  eval_mode            offline-proxy
  cache_hit_rate       0.65
  cache_hit_accuracy   0.8462
  cache_miss_accuracy  0.7143
  n                    20

reportable: 6 numeric + 2 N/A


## 2 · Persist per-query cost/latency — `scripts/init_results_db.py`

At the end of W10, `results.db` held only the semantic-cache tables. W11 adds one small `eval_results` table logging per-query cost/latency/cache-hit **tagged by `kb_version`**, so the snapshot is reproducible without a live re-run — and so you never aggregate across two KB versions by accident.

In [8]:
"""scripts/init_results_db.py — add an eval-results table to results.db.

Reality check from the plan: at end of W10, `results.db` holds only the
semantic-cache tables (`query_cache`, `query_cache_meta`). The curriculum
says cost/latency come "from results.db", so W11 adds one small table that
logs per-query cost/latency/cache-hit for an eval run. test_kpi.py reads from
it when present (and falls back to the in-memory rows otherwise), so the
snapshot is reproducible without a live re-run.

Usage:
    python3 scripts/init_results_db.py            # create the table
    # then run_eval.py --persist writes rows into it
"""
from __future__ import annotations

import sqlite3
import sys
from pathlib import Path

DEFAULT_DB = "./results.db"

SCHEMA = """
CREATE TABLE IF NOT EXISTS eval_results (
    run_label   TEXT    NOT NULL,
    kb_version  TEXT    NOT NULL,
    golden_id   TEXT    NOT NULL,
    cache_hit   INTEGER NOT NULL,
    latency_ms  INTEGER NOT NULL,
    cost_usd    REAL    NOT NULL,
    created_at  TEXT    DEFAULT (datetime('now')),
    PRIMARY KEY (run_label, golden_id)
);
"""


def init_db(db_path: str = DEFAULT_DB) -> None:
    conn = sqlite3.connect(db_path)
    try:
        conn.executescript(SCHEMA)
        conn.commit()
    finally:
        conn.close()


def write_rows(rows: list[dict], run_label: str, kb_version: str,
               db_path: str = DEFAULT_DB) -> int:
    """Persist per-query cost/latency for an eval run. Returns rows written."""
    init_db(db_path)
    conn = sqlite3.connect(db_path)
    try:
        conn.executemany(
            "INSERT OR REPLACE INTO eval_results "
            "(run_label, kb_version, golden_id, cache_hit, latency_ms, cost_usd) "
            "VALUES (?,?,?,?,?,?)",
            [(run_label, kb_version, r["id"], int(bool(r.get("cache_hit"))),
              int(r.get("latency_ms", 0)), float(r.get("cost_usd", 0.0)))
             for r in rows],
        )
        conn.commit()
        return len(rows)
    finally:
        conn.close()

# (The module's command-line entry point is intentionally omitted in the notebook.
#  Call init_db(...) and write_rows(...) directly instead, as the next cell does.)


**Write this run's rows, then read them back** (grouped by `kb_version`):

In [9]:
import sqlite3
init_db('results.db')
n = write_rows(rows, run_label='wk11-baseline', kb_version=kb_version, db_path='results.db')
print('persisted', n, 'rows')

conn = sqlite3.connect('results.db')
q = conn.execute('SELECT kb_version, COUNT(*), ROUND(AVG(cost_usd),6), ROUND(AVG(latency_ms),1) '
                 'FROM eval_results GROUP BY kb_version').fetchall()
conn.close()
print('kb_version | rows | avg cost | avg latency')
for row in q:
    print('  ', row)

persisted 20 rows
kb_version | rows | avg cost | avg latency
   ('v2.w10', 20, 0.000171, 394.4)


## 3 · The baseline report — `scripts/run_eval.py`

The *produce-real-numbers-on-demand* entry point renders a markdown baseline report. Offline output is **banner-marked as proxy numbers** — never paste those into the M2 dashboard; run live for the numbers you defend. Here is its report renderer verbatim, called directly (the notebook replaces the module's `argparse` CLI wrapper).

In [10]:
def _render_report(rows, kpis, extras, metrics) -> str:
    mode = extras["eval_mode"]
    banner = ""
    if mode == "offline-proxy":
        banner = (
            "> ⚠️ **MODE: offline-proxy.** These are deterministic token-overlap "
            "proxies for CI, **not** RAGAS numbers. Run live (`python3 "
            "scripts/run_eval.py`) for the numbers you put in the snapshot.\n\n"
        )
    lines = [
        "# wk11 RAGAS baseline (auto-generated)\n",
        banner,
        f"- **eval mode:** `{mode}`\n",
        f"- **kb_version:** `{extras['kb_version']}`\n",
        f"- **golden set:** {extras['n']} entries\n",
        "\n## RAG metrics (RAGAS)\n",
        "| Metric | Aggregate |\n|---|---|\n",
    ]
    for name in ("faithfulness", "answer_relevance",
                 "context_precision", "context_recall"):
        lines.append(f"| {name} | {metrics[name].aggregate:.3f} |\n")

    lines.append("\n## 8-KPI dashboard\n")
    lines.append("| # | KPI | Value |\n|---|---|---|\n")
    rowmap = [
        ("1", "Task success rate",     kpis["1_task_success_rate"]),
        ("2", "Grounded response rate", kpis["2_grounded_response_rate"]),
        ("3", "Hallucination rate",    kpis["3_hallucination_rate"]),
        ("4", "Retrieval hit rate",    kpis["4_retrieval_hit_rate"]),
        ("5", "Tool success rate",     kpis["5_tool_success_rate"]),
        ("6", "Cost per query (USD)",  kpis["6_cost_per_query_usd"]),
        ("7", "Latency p50 / p95 (ms)",
         f"{kpis['7_latency_p50_ms']:.0f} / {kpis['7_latency_p95_ms']:.0f}"),
        ("8", "Escalation rate",       kpis["8_escalation_rate"]),
    ]
    for num, label, val in rowmap:
        lines.append(f"| {num} | {label} | {val} |\n")

    lines.append("\n## W11 extensions (W10 tombstone payoff)\n")
    lines.append(
        f"- cache hit rate: {extras['cache_hit_rate']}\n"
        f"- cache-hit accuracy: {extras['cache_hit_accuracy']} · "
        f"cache-miss accuracy: {extras['cache_miss_accuracy']}\n"
        f"- results tagged `kb_version={extras['kb_version']}`\n"
    )

    lines.append("\n## Per-entry (lowest faithfulness first)\n")
    faith = metrics["faithfulness"].per_row
    lines.append("| id | faithfulness | notes |\n|---|---|---|\n")
    order = sorted(rows, key=lambda r: faith.get(r["id"], 0.0))
    for r in order[:6]:
        note = (r.get("notes", "") or "").split(".")[0][:48]
        lines.append(f"| {r['id']} | {faith[r['id']]:.3f} | {note} |\n")
    return "".join(lines)


**Render and display the report:**

In [11]:
from IPython.display import Markdown, display
metrics = out['metrics']
report = _render_report(rows, kpis, extras, metrics)
display(Markdown(report))

# wk11 RAGAS baseline (auto-generated)
> ⚠️ **MODE: offline-proxy.** These are deterministic token-overlap proxies for CI, **not** RAGAS numbers. Run live (`python3 scripts/run_eval.py`) for the numbers you put in the snapshot.

- **eval mode:** `offline-proxy`
- **kb_version:** `v2.w10`
- **golden set:** 20 entries

## RAG metrics (RAGAS)
| Metric | Aggregate |
|---|---|
| faithfulness | 0.925 |
| answer_relevance | 0.731 |
| context_precision | 0.667 |
| context_recall | 0.944 |

## 8-KPI dashboard
| # | KPI | Value |
|---|---|---|
| 1 | Task success rate | 0.8 |
| 2 | Grounded response rate | 0.9252 |
| 3 | Hallucination rate | 0.0748 |
| 4 | Retrieval hit rate | 0.6667 |
| 5 | Tool success rate | N/A (W13 — no tools yet) |
| 6 | Cost per query (USD) | 0.000171 |
| 7 | Latency p50 / p95 (ms) | 184 / 785 |
| 8 | Escalation rate | N/A (W17 — no HITL yet) |

## W11 extensions (W10 tombstone payoff)
- cache hit rate: 0.65
- cache-hit accuracy: 0.8462 · cache-miss accuracy: 0.7143
- results tagged `kb_version=v2.w10`

## Per-entry (lowest faithfulness first)
| id | faithfulness | notes |
|---|---|---|
| g005 | 0.318 | happy-path |
| g015 | 0.556 | harder · multi-hop |
| g019 | 0.630 | edge |
| g001 | 1.000 | happy-path |
| g002 | 1.000 | happy-path |
| g003 | 1.000 | happy-path |


## 4 · Thresholds from a baseline

A threshold on the floor tests nothing. Read your baseline aggregate, set the bar **just below it** — a real regression trips it; run-to-run judge noise does not — and record the number *and the reason* in the ADR. This is exactly what the pytest suite asserts on every commit; the same check, inline:

In [12]:
# baseline-set bars (see the deck / ADR): set just below the live baseline aggregate
THRESHOLDS = {'faithfulness': 0.85, 'answer_relevance': 0.85,
              'context_precision': 0.75, 'context_recall': 0.80}

print('metric              aggregate   bar     pass?')
for name, bar in THRESHOLDS.items():
    agg = metrics[name].aggregate
    print(f'  {name:18s} {agg:.3f}      {bar:.2f}    {agg >= bar}')

# (offline proxies won't all clear live-set bars — that's expected; the point is the
#  wiring. Run live, then set each bar just below the real aggregate.)

metric              aggregate   bar     pass?
  faithfulness       0.925      0.85    True
  answer_relevance   0.731      0.85    False
  context_precision  0.667      0.75    False
  context_recall     0.944      0.80    True


## Where this graduates to

These notebooks are the place to **see and explain** the eval — run a cell, read a number, talk to it. The one thing a notebook can't do is gate a deploy or run on every commit, so in the package this same code lives as a **pytest suite** (`eval/test_*.py`) that runs offline in CI and produces the live numbers on demand. Notebook to understand it; the suite to enforce it. Next week (W12) is the eval regression runbook and DR #2, where you present these numbers and defend your thresholds.